<a href="https://colab.research.google.com/github/ks148678-beep/Data-Warehousing-and-Data-Mining/blob/main/cleaning_patient_dataipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!wgethttps://raw.githubusercontent.com/ks148678-beep/Data-Warehousing-and-Data-Mining/refs/heads/main/clinic_patients_dirty.csv

/bin/bash: line 1: wgethttps://raw.githubusercontent.com/ks148678-beep/Data-Warehousing-and-Data-Mining/refs/heads/main/clinic_patients_dirty.csv: No such file or directory


In [21]:
import numpy as np
import pandas as pd
import os

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# Automatically fall back to the existing copy with duplicate suffixes if the exact file isn't present
if os.path.exists("clinic_patients_dirty.csv"):
    DATA_PATH = "clinic_patients_dirty.csv"
elif os.path.exists("clinic_patients_dirty (6).csv"):
    DATA_PATH = "clinic_patients_dirty (6).csv"
else:
    DATA_PATH = "clinic_patients_dirty.csv"

OUT_PATH = "clinic_patients_clean.csv"

log = []
def record(step, action, n):
    log.append({"step": step, "action": action, "affected": int(n)})
    print(f" -> {action}: {int(n)}")

raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
df = raw.copy()
print("Shape:", df.shape)
print("\nColumn names:", list(df.columns))
print(df.head(3).to_string())

Shape: (420, 15)

Column names: ['Patient ID', ' Full Name', 'SEX', 'Age (yrs)', 'County ', 'Visit_Date', 'Height', 'Weight (kg)', 'BMI', 'Systolic BP', 'diastolic bp', 'Temp', 'Blood Group', 'Diagnosis ', 'Smoker?']
  Patient ID      Full Name     SEX Age (yrs)   County   Visit_Date  Height Weight (kg)   BMI Systolic BP diastolic bp  Temp Blood Group    Diagnosis  Smoker?
0    PT-0165  Purity Omondi  Female        19  Machakos  2025-06-09     164        83.8  21.7         142            ?  36.9          O+  Hypertension        
1    PT-0350  Halima Barasa  Female        45    Kiambu  01.06.2025     160        44.5  17.4         119           77  38.9          B+       Malaria     Yes
2    PT-0395   david otieno    Male        27   Nairobi  2025-10-27  177 cm        79.1               143           96  38.2          o+  Hypertension      No


In [22]:
df.columns = (
    df.columns.str.strip()
   .str.lower()
   .str.replace(r"\(.*?\)", "", regex=True)
   .str.strip()
   .str.replace(r"[^a-z0-9]+", "_", regex=True)
   .str.strip("_")
)
# Fix names
df = df.rename(columns={
    "full_name": "patient_name",
    "age_yrs": "age",
    "visit_date": "visit_date",
    "weight_kg": "weight",
    "blood_group": "blood_group",
    "smoker": "smoker",
    "height": "height",
    "bmi": "bmi",
    "systolic_bp": "systolic_bp",
    "diastolic_bp": "diastolic_bp",
    "temp": "temp"
})
print("Clean names:", list(df.columns))

Clean names: ['patient_id', 'patient_name', 'sex', 'age', 'county', 'visit_date', 'height', 'weight', 'bmi', 'systolic_bp', 'diastolic_bp', 'temp', 'blood_group', 'diagnosis', 'smoker']


In [23]:
# STEP 3: Strip whitespace + expose placeholders
before = df.copy()
df = df.apply(lambda s: s.str.strip() if s.dtype == object else s)
record(3, "whitespace trimmed", (before!= df).sum().sum())

MISSING_TOKENS = {"", "n/a", "na", "null", "none", "nan", "?", "-", "unknown", "not recorded"}
is_missing = df.apply(lambda s: s.str.lower().isin(MISSING_TOKENS) if s.dtype==object else pd.Series(False, index=s.index))
print("\nPlaceholders:\n", df.where(is_missing).stack().value_counts().to_string())
df = df.mask(is_missing)
record(3, "placeholders -> NaN", is_missing.sum().sum())

print("\nMissing after exposing:\n", df.isna().sum().to_string())

 -> whitespace trimmed: 102

Placeholders:
                 165
NULL             23
?                21
n/a              20
N/A              20
-                19
not recorded     15
unknown          13
 -> placeholders -> NaN: 296

Missing after exposing:
 patient_id       0
patient_name     0
sex             17
age             22
county          21
visit_date      14
height          16
weight          22
bmi             23
systolic_bp     19
diastolic_bp    37
temp            17
blood_group     42
diagnosis       13
smoker          33


In [34]:
# STEP 4: Standardise patient_id and remove duplicates
print("Duplicates before:", df.duplicated().sum())
df["patient_id"] = df["patient_id"].str.upper().str.replace(r"[\s_]+", "-", regex=True).str.strip()
n = len(df)
df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
record(4, "duplicate patient_id removed", n-len(df))

# STEP 5: Categorical cleaning
print("sex BEFORE:", sorted(df["sex"].dropna().unique()))
df["sex"] = df["sex"].str.lower().str.strip().map(
    {"m":"Male","male":"Male","f":"Female","female":"Female"}
)
print("sex AFTER:", sorted(df["sex"].dropna().unique()))

# County
city = df["county"].str.lower().str.replace(r"[^a-z ]","",regex=True).str.strip()
city = city.replace({"nairobbi":"nairobi","machackos":"machakos","nrb":"nairobi","mombassa":"mombasa"})
df["county"] = city.str.title()

# Blood group
bg = df["blood_group"].str.upper().str.replace(r"\s+","",regex=True)
bg = bg.replace({"OPOS":"O+","BPOS":"B+","APOS":"A+","ONEG":"O-","ANEG":"A-","BNEG":"B-","ABPOS":"AB+"})
df["blood_group"] = bg

# Diagnosis
diag = df["diagnosis"].str.lower().str.strip()
diag = diag.replace({
    "maleria":"malaria","high blood pressure":"hypertension",
    "type 2 diabetes":"diabetes","urti":"upper respiratory tract infection",
    "pneumonia":"pneumonia"
})
df["diagnosis"] = diag.str.title()

# Smoker
sm = df["smoker"].str.lower().map({"yes":"Yes","y":"Yes","true":"Yes","no":"No","n":"No","false":"No"})
df["smoker"] = sm

# Names
df["patient_name"] = df["patient_name"].str.title().str.replace(r"\s+"," ",regex=True)

Duplicates before: 0
 -> duplicate patient_id removed: 0
sex BEFORE: ['Female', 'Male', 'Unknown']
sex AFTER: ['Female', 'Male']


In [25]:
# STEP 6: Convert types
# Height: "177 cm", "1.62" -> 162 cm
h = df["height"].astype(str)
# if value like 1.62 (meters) convert to cm
def parse_height(x):
    try:
        if pd.isna(x): return np.nan
        x=str(x).lower().replace("cm","").strip()
        v=float(x)
        if v < 3: # meters like 1.62
            v = v*100
        return v
    except: return np.nan

df["height"] = df["height"].apply(parse_height)

# Weight: "63.8 kg"
df["weight"] = pd.to_numeric(df["weight"].astype(str).str.replace(r"[^0-9.\-]","",regex=True), errors="coerce")

# Temp: handle F to C, e.g. 98.6F -> 37C
def parse_temp(x):
    if pd.isna(x): return np.nan
    s=str(x).lower()
    try:
        num = float(''.join(c for c in s if c.isdigit() or c=='.' or c=='-'))
        if 'f' in s: # Fahrenheit
            return round((num-32)*5/9,1)
        return num
    except: return np.nan
df["temp"] = df["temp"].apply(parse_temp)

# BP: "102/66" -> split, "-1" invalid
def parse_sys(x):
    if pd.isna(x): return np.nan
    s=str(x)
    if '/' in s:
        s=s.split('/')[0]
    try:
        v=float(s)
        return v if v>0 else np.nan
    except: return np.nan
df["systolic_bp"] = df["systolic_bp"].apply(parse_sys)
df["diastolic_bp"] = pd.to_numeric(df["diastolic_bp"], errors="coerce")
df["diastolic_bp"] = df["diastolic_bp"].apply(lambda v: v if pd.isna(v) or v>0 else np.nan)

df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")
df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")

# Date: 4 layouts like retail
date_txt = df["visit_date"]
parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for fmt in ["%d/%m/%Y","%m/%d/%Y","%d.%m.%Y","%d-%b-%y","%Y-%m-%d"]:
    attempt = pd.to_datetime(date_txt, format=fmt, errors="coerce")
    parsed = parsed.fillna(attempt)
df["visit_date"] = parsed
print(df.dtypes)

patient_id              object
patient_name            object
sex                     object
age                      Int64
county                  object
visit_date      datetime64[ns]
height                 float64
weight                 float64
bmi                    float64
systolic_bp            float64
diastolic_bp           float64
temp                   float64
blood_group             object
diagnosis               object
smoker                  object
dtype: object


In [26]:
# STEP 7: Domain rules
df.loc[~df["age"].between(0,100), "age"] = pd.NA
record(7, "invalid age -> NaN", 0)
df.loc[~df["height"].between(100,220), "height"] = np.nan
df.loc[~df["weight"].between(20,200), "weight"] = np.nan
df.loc[~df["temp"].between(35,42), "temp"] = np.nan

# STEP 8 & 9: Repair BMI = weight / (height_m)^2
# Recalculate BMI where possible
height_m = df["height"]/100
expected_bmi = df["weight"] / (height_m**2)
# If BMI missing but we have height+weight, recover
mask = df["bmi"].isna() & expected_bmi.notna()
df.loc[mask, "bmi"] = expected_bmi[mask].round(1)
record(9, "BMI recovered from height/weight", mask.sum())

# STEP 10: Impute remaining
df["age"] = df["age"].fillna(int(df["age"].median()))
df["height"] = df["height"].fillna(df["height"].median())
df["weight"] = df["weight"].fillna(df["weight"].median())
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df["systolic_bp"] = df["systolic_bp"].fillna(df["systolic_bp"].median())
df["diastolic_bp"] = df["diastolic_bp"].fillna(df["diastolic_bp"].median())
df["temp"] = df["temp"].fillna(df["temp"].median())

for col in ["sex","county","blood_group","diagnosis","smoker"]:
    df[col] = df[col].fillna("Unknown")

# Total rebuilt
df["bmi"] = (df["weight"] / ((df["height"]/100)**2)).round(1)

# STEP 11: Export
df = df.sort_values("patient_id").reset_index(drop=True)
df.to_csv(OUT_PATH, index=False)
print(f"Clean saved to {OUT_PATH}, Rows: {len(raw)} -> {len(df)}")
print(pd.DataFrame(log).to_string(index=False))
df.head()

 -> invalid age -> NaN: 0
 -> BMI recovered from height/weight: 20
Clean saved to clinic_patients_clean.csv, Rows: 420 -> 400
 step                           action  affected
    3               whitespace trimmed       102
    3              placeholders -> NaN       296
    4     duplicate patient_id removed        20
    7               invalid age -> NaN         0
    9 BMI recovered from height/weight        20


,patient_id,patient_name,sex,age,county,visit_date,height,weight,bmi,systolic_bp,diastolic_bp,temp,blood_group,diagnosis,smoker
0,PT-0001,George Otieno,Male,70,Nairobi,1925-05-20,177.0,84.1,26.8,117.0,69.0,37.4,A-,Unknown,Yes
1,PT-0002,Ruth Kiplagat,Female,44,Kiambu,2025-10-02,159.0,76.3,30.2,119.0,71.0,36.6,A+,Diabetes,No
2,PT-0003,Lucy Kariuki,Female,55,Nakuru,2025-05-14,153.0,60.2,25.7,106.0,78.0,37.4,Unknown,Malaria,Yes
3,PT-0004,Samuel Abdi,Male,32,Nakuru,2025-06-07,176.0,88.3,28.5,123.0,79.0,37.4,A+,Upper Respiratory Tract Infection,No
4,PT-0005,Tabitha Kariuki,Female,78,Nairobi,2025-07-20,173.0,84.7,28.3,113.0,72.0,38.4,B+,Pneumonia,No


In [37]:
# Use Colab's built-in interactive table extension (no Sheets API authentication required)
from google.colab import data_table

# Configure to show 10 rows per page
data_table.enable_dataframe_formatter()

# Display the cleaned DataFrame
df

,patient_id,patient_name,sex,age,county,visit_date,height,weight,bmi,systolic_bp,diastolic_bp,temp,blood_group,diagnosis,smoker
0,PT-0001,George Otieno,Male,70,Nairobi,1925-05-20,177.0,84.1,26.8,117.0,69.0,37.4,A-,Unknown,Yes
1,PT-0002,Ruth Kiplagat,Female,44,Kiambu,2025-10-02,159.0,76.3,30.2,119.0,71.0,36.6,A+,Diabetes,No
2,PT-0003,Lucy Kariuki,Female,55,Nakuru,2025-05-14,153.0,60.2,25.7,106.0,78.0,37.4,UNKNOWN,Malaria,Yes
3,PT-0004,Samuel Abdi,Male,32,Nakuru,2025-06-07,176.0,88.3,28.5,123.0,79.0,37.4,A+,Upper Respiratory Tract Infection,No
4,PT-0005,Tabitha Kariuki,Female,78,Nairobi,2025-07-20,173.0,84.7,28.3,113.0,72.0,38.4,B+,Pneumonia,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,PT-0396,Esther Mwangi,Female,43,Nairobi,2025-03-05,155.0,62.8,26.1,123.0,81.0,36.3,A-,Upper Respiratory Tract Infection,No
396,PT-0397,Wanjiru Omondi,Female,49,Nakuru,NaT,159.0,53.6,21.2,144.0,94.0,38.7,O-,Typhoid Fever,No
397,PT-0398,David Kiplagat,Male,40,Nairobi,2025-12-26,187.0,81.0,23.2,125.0,78.0,38.2,O+,Pneumonia,No
398,PT-0399,Kevin Omondi,Male,34,Machakos,2025-10-31,168.0,65.2,23.1,114.0,75.0,38.3,A+,Typhoid,No
